# 11 Download Statcast And Video Sources

目的: smoke sample ではなく、実際の Baseball Savant / Statcast BBE と exact playId 優先の video candidate を Colab / Drive 上に作ります。

この notebook は **動画ファイルの download までは行いません**。ここで作るのは次の artifact までです。

- `/content/drive/MyDrive/baseball_vision/raw_statcast/`
- `/content/drive/MyDrive/baseball_vision/manifests/bbe_events_v1.parquet`
- `/content/drive/MyDrive/baseball_vision/manifests/video_sources_v1.parquet`
- `/content/drive/MyDrive/baseball_vision/reports/preflight/statsapi_video_resolve_progress_v1.json`

`video_sources_v1.parquet` ができたら、この notebook は終了です。次は `11_a_download_video_shard_1.ipynb`, `11_b_download_video_shard_2.ipynb`, `11_c_download_video_shard_3.ipynb` を別 runtime / tab で並列実行し、最後に `11_d_merge_video_download_shards.ipynb` で merge します。

In [ ]:
from pathlib import Path
import os
import sys
import importlib.util


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path('/content/drive/MyDrive/codex/batting_codex_handoff')
    candidates: list[Path] = []
    env_root = os.environ.get('BATTING_CODE_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend(
        [
            fixed,
            Path.cwd(),
        ]
    )
    candidates.extend(Path.cwd().parents)

    checked: list[str] = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            if candidate != fixed:
                print('WARNING: fixed REPO_DIR ではない場所の repo を使います。')
                print('  fixed:', fixed)
                print('  using:', candidate)
                print('  次回は repo フォルダを fixed path に置くか、BATTING_CODE_ROOT を明示してください。')
            return candidate

    raise FileNotFoundError(
        'sport_pipeline package が見つかりません。Drive には notebook 単体ではなく repo フォルダごと置く必要があります。\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別の場所に置いた場合は、この notebook の最初の cell より前に次を実行してください。\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/batting_codex_handoff\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()

REPO_DIR = _resolve_repo_dir()
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ.get('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME

BASE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(
        'sport_pipeline を import できません。repo 配置と src/sport_pipeline/__init__.py を確認してください。\n'
        f'REPO_DIR={REPO_DIR}\n'
        f'src_dir={src_dir}\n'
        f'src_dir_exists={src_dir.exists()}'
    )

print('REPO_DIR =', REPO_DIR)
print('BASE_DIR =', BASE_DIR)
print('CACHE_DIR =', CACHE_DIR)
print('RUN_PROFILE_PATH =', RUN_PROFILE_PATH)
print('src_dir =', src_dir)

from sport_pipeline.pipeline.run_profile import load_run_profile, resolve_statcast_date_range, run_id, stage_settings, threshold

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
START_DATE, END_DATE = resolve_statcast_date_range(RUN_PROFILE)
RUN_ID = run_id(RUN_PROFILE, 'full_run_id')

print('STATCAST_DATE_RANGE =', START_DATE, 'to', END_DATE)
print('RUN_ID =', RUN_ID)


## Statcast Download

まず小さな日付範囲で pilot します。成功したら範囲を広げます。母集団は BBE event universe であり、home run topic ではありません。

In [ ]:
from collections import Counter
from sport_pipeline.statcast.download import download_statcast_bbe_dataset

STATCAST_SETTINGS = stage_settings(RUN_PROFILE, 'statcast_download')
ENABLE_STATCAST_DOWNLOAD = bool(STATCAST_SETTINGS.get('execute_default', True))
CHUNK_DAYS = int(RUN_PROFILE.get('data_window', {}).get('chunk_days', 1))
GAME_TYPE = str(RUN_PROFILE.get('data_window', {}).get('game_type', 'R'))
TIMEOUT_SEC = int(STATCAST_SETTINGS.get('timeout_sec', 180))
RESUME_DOWNLOAD = bool(STATCAST_SETTINGS.get('resume', True))
REDOWNLOAD_EMPTY = bool(STATCAST_SETTINGS.get('redownload_empty', True))
ALLOW_EMPTY_MANIFEST = bool(STATCAST_SETTINGS.get('allow_empty_manifest', False))
ALLOW_PARTIAL_MANIFEST = bool(STATCAST_SETTINGS.get('allow_partial_manifest', False))

statcast_summary = download_statcast_bbe_dataset(
    base_dir=BASE_DIR,
    start_date=START_DATE,
    end_date=END_DATE,
    execute=ENABLE_STATCAST_DOWNLOAD,
    chunk_days=CHUNK_DAYS,
    game_type=GAME_TYPE,
    build_manifest=bool(STATCAST_SETTINGS.get('build_manifest', True)),
    timeout_sec=TIMEOUT_SEC,
    resume=RESUME_DOWNLOAD,
    redownload_empty=REDOWNLOAD_EMPTY,
    allow_empty_manifest=ALLOW_EMPTY_MANIFEST,
    allow_partial_manifest=ALLOW_PARTIAL_MANIFEST,
)

print('execute =', ENABLE_STATCAST_DOWNLOAD)
print('date_range =', START_DATE, 'to', END_DATE)
print('resume =', RESUME_DOWNLOAD, 'redownload_empty =', REDOWNLOAD_EMPTY, 'allow_empty_manifest =', ALLOW_EMPTY_MANIFEST, 'allow_partial_manifest =', ALLOW_PARTIAL_MANIFEST)
print('progress_path =', statcast_summary['progress_path'])
print('summary_path =', statcast_summary['summary_path'])
print('filtered_bbe_csv =', statcast_summary['filtered_bbe_csv'])
print('manifest_outputs =', statcast_summary['manifest_outputs'])
print('filter_summary =', statcast_summary.get('filter_summary'))
if statcast_summary.get('warnings'):
    print('WARNINGS:')
    for warning in statcast_summary['warnings']:
        print('-', warning)
status_counts = Counter(chunk.get('status') for chunk in statcast_summary['chunks'])
print('chunk_status_counts =', dict(status_counts))
for chunk in statcast_summary['chunks'][:10]:
    print(
        chunk['start_date'], chunk['end_date'], chunk['status'],
        'rows=', chunk.get('rows_downloaded'),
        'bbe_rows=', chunk.get('bbe_rows'),
        'size=', chunk.get('size_bytes'),
        chunk['raw_csv_path'],
    )

if ENABLE_STATCAST_DOWNLOAD:
    failed_chunks = statcast_summary.get('failed_chunks') or []
    if failed_chunks and not ALLOW_PARTIAL_MANIFEST:
        raise RuntimeError(
            f'Statcast download has {len(failed_chunks)} failed chunks; manifest build was skipped to avoid a partial event universe. '
            f'Progress was saved at {statcast_summary["progress_path"]}. Re-run this cell; cached successful chunks will be reused.'
        )
    bbe_rows = int((statcast_summary.get('filter_summary') or {}).get('bbe_rows') or 0)
    raw_rows = int((statcast_summary.get('filter_summary') or {}).get('raw_rows') or 0)
    min_bbe_rows = threshold(RUN_PROFILE, 'min_bbe_rows', 0)
    if bbe_rows < min_bbe_rows:
        raise RuntimeError(
            f'Statcast BBE rows too small for real run: {bbe_rows} < {min_bbe_rows}. '
            f'raw_rows={raw_rows}. Progress was saved at {statcast_summary["progress_path"]}. '
            'If many chunks show rows=0, rerun this cell after updating the repo; empty cached chunks are now redownloaded. '
            'If rows still stay 0, open one raw_statcast/statcast_YYYY-MM-DD_to_YYYY-MM-DD.csv and inspect its header/content.'
        )


## MLB Game Content Video Candidate Resolve

`bbe_events_v1.parquet` の `play_id` から Baseball Savant exact-play video を優先して解決し、補助的に MLB Stats API game content を見ます。動画 candidate を `video_sources_v1.parquet` に保存します。これは candidate であり、clean clip ではありません。

In [ ]:
from collections import Counter

from sport_pipeline.io import read_table
from sport_pipeline.video.mlb_statsapi import build_mlb_statsapi_video_source_artifact
from sport_pipeline.video.source_quality import is_event_level_video_source

RESOLVE_SETTINGS = stage_settings(RUN_PROFILE, 'statsapi_resolve')
ENABLE_STATSAPI_RESOLVE = bool(RESOLVE_SETTINGS.get('execute_default', True))
MAX_ASSETS_PER_EVENT = int(RESOLVE_SETTINGS.get('max_assets_per_event', 2))
MIN_MATCH_CONFIDENCE = float(RESOLVE_SETTINGS.get('min_match_confidence', 0.45))
TIMEOUT_SEC = int(RESOLVE_SETTINGS.get('timeout_sec', 60))
MAX_EVENTS = RESOLVE_SETTINGS.get('max_events')
MAX_SAVANT_PLAY_RESOLVES = RESOLVE_SETTINGS.get('max_savant_play_resolves')
INCLUDE_SAVANT_PLAY_RESOLVER = bool(RESOLVE_SETTINGS.get('include_savant_play_resolver', True))
RESUME_RESOLVE = bool(RESOLVE_SETTINGS.get('resume', True))
CHECKPOINT_EVERY_EVENTS = int(RESOLVE_SETTINGS.get('checkpoint_every_events', 25))

bbe_events = BASE_DIR / 'manifests' / 'bbe_events_v1.parquet'
if not bbe_events.exists():
    raise FileNotFoundError(f'bbe_events_v1 がありません。上の Statcast download を execute=True で実行してください: {bbe_events}')

if ENABLE_STATSAPI_RESOLVE:
    video_sources_path = build_mlb_statsapi_video_source_artifact(
        base_dir=BASE_DIR,
        bbe_path=bbe_events,
        cache_dir=CACHE_DIR / 'statsapi',
        max_assets_per_event=MAX_ASSETS_PER_EVENT,
        min_match_confidence=MIN_MATCH_CONFIDENCE,
        timeout_sec=TIMEOUT_SEC,
        include_savant_play_resolver=INCLUDE_SAVANT_PLAY_RESOLVER,
        max_events=MAX_EVENTS,
        max_savant_play_resolves=MAX_SAVANT_PLAY_RESOLVES,
        resume=RESUME_RESOLVE,
        checkpoint_every_events=CHECKPOINT_EVERY_EVENTS,
    )
    rows = read_table(video_sources_path)
    media_rows = [row for row in rows if row.get('media_url')]
    event_level_media_rows = [row for row in media_rows if is_event_level_video_source(row, min_match_confidence=MIN_MATCH_CONFIDENCE)]
    print('video_sources ->', video_sources_path)
    print('resume =', RESUME_RESOLVE, 'checkpoint_every_events =', CHECKPOINT_EVERY_EVENTS)
    print('progress_path =', BASE_DIR / 'reports/preflight/statsapi_video_resolve_progress_v1.json')
    print('video_source_rows =', len(rows), 'media_url_rows =', len(media_rows))
    print('event_level_media_rows =', len(event_level_media_rows), '(download/CV training gate)')
    print('source_kind_counts =', dict(Counter(row.get('source_kind') for row in rows)))
    print('top_match_reason_counts =', dict(Counter(row.get('match_reason') for row in rows).most_common(20)))
    min_video_sources = threshold(RUN_PROFILE, 'min_video_sources', 0)
    min_media_url_rows = threshold(RUN_PROFILE, 'min_media_url_rows', 0)
    if len(rows) < min_video_sources:
        raise RuntimeError(f'video_sources rows too small for real run: {len(rows)} < {min_video_sources}')
    if len(media_rows) < min_media_url_rows:
        raise RuntimeError(f'direct media_url rows too small for real run: {len(media_rows)} < {min_media_url_rows}')
else:
    print('ENABLE_STATSAPI_RESOLVE=False のため video source resolve は実行していません。')
    print('既存 video_sources:', BASE_DIR / 'manifests' / 'video_sources_v1.parquet')


## Stop Here, Then Move To 11_a / 11_b / 11_c

`video_sources_v1.parquet` が作成できていれば、この notebook の役割は完了です。動画downloadはこの notebook ではなく、次の3つの shard notebook で行います。

In [ ]:
from sport_pipeline.io import read_table

video_sources = BASE_DIR / 'manifests' / 'video_sources_v1.parquet'
if not video_sources.exists():
    raise FileNotFoundError(f'video_sources_v1.parquet がありません。上の StatsAPI resolve cell を先に成功させてください: {video_sources}')

rows = read_table(video_sources)
media_rows = [row for row in rows if row.get('media_url')]
print('11 completed: video source candidate manifest is ready.')
print('video_sources ->', video_sources)
print('video_source_rows =', len(rows), 'media_url_rows =', len(media_rows))
print('')
print('NEXT: open these three notebooks in separate Colab runtime/tab if possible:')
print('  1. notebooks/11_a_download_video_shard_1.ipynb')
print('  2. notebooks/11_b_download_video_shard_2.ipynb')
print('  3. notebooks/11_c_download_video_shard_3.ipynb')
print('')
print('After all three finish, run:')
print('  notebooks/11_d_merge_video_download_shards.ipynb')
print('')
print('Do not continue downloading videos in this notebook; it intentionally stops here.')